# Append Transition SoC to Diagnostic Summaries

This notebook appends the silicon-dominant transition SoC to every diagnostic summary CSV generated by the lifetime and C-rate workflows. It automatically scans for both supported summary formats: per-cell lifetime files (`*_voltage_fit_summary.csv`) and structured C-rate files (`structured_batch_esoh_summary.csv`). No manual pattern switch is required.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile
import traceback

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Return the repository root that contains the shared code and data folders."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
CODE_DIR = REPO_ROOT / "code"
PROJECT_DIR = CODE_DIR / "diagnostic_algorithm_lifetime_crate"
ROOT_DIR = PROJECT_DIR / "batch_results"

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "managing_si_burnout_matplotlib"))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from diagnostic_algorithm_lifetime_crate.user_functions import Gr_OCP, build_effective_si_ocp, ocp_3
from diagnostic_algorithm_lifetime_crate.transition_soc_utils import (
    append_transition_soc_to_summary_df,
    compute_si_current_share_and_transition_7p,
    save_transition_soc_plot,
    smooth_ocp_curve,
)
from diagnostic_algorithm_lifetime_crate.derivative_utils import smooth_then_grad

Gr_OCP_smooth = smooth_ocp_curve(
    Gr_OCP,
    x_col="sto",
    y_col="p",
    window_length=51,
    polyorder=3,
    enforce_monotone=True,
)

Qdata_expand = np.linspace(-3, 3, 600)

print("Repository root:", REPO_ROOT)
print("Batch-results folder:", ROOT_DIR)


In [ ]:
# Automatically scan both summary formats produced by this folder.
SUMMARY_PATTERNS = (
    "*_voltage_fit_summary.csv",      # lifetime diagnostics: one file per cell
    "structured_batch_esoh_summary.csv",  # C-rate diagnostics: one file per run
)

OVERWRITE_CSV = True
SAVE_BACKUP = True
DRY_RUN = False

# Transition-SoC definition used in the paper: first persistent drop of silicon
# current share below 50%, checked over the following 15% SoC interval.
SI_RECONSTRUCTION_MODE = "reconstructed"
WIN = 3
POLY = 3
THRESHOLD = 0.5
PERSISTENCE_WINDOW = 0.15
TOL = 0.01

SAVE_TRANSITION_PLOTS = True


In [ ]:
required_objs = [
    "append_transition_soc_to_summary_df",
    "compute_si_current_share_and_transition_7p",
    "save_transition_soc_plot",
    "Qdata_expand",
    "ocp_3",
    "smooth_then_grad",
    "build_effective_si_ocp",
    "Gr_OCP_smooth",
]
missing_objs = [name for name in required_objs if name not in globals()]
if missing_objs:
    raise RuntimeError("Missing required objects: " + ", ".join(missing_objs))

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"ROOT_DIR does not exist: {ROOT_DIR}")

matched = []
for pattern in SUMMARY_PATTERNS:
    for csv_path in ROOT_DIR.rglob(pattern):
        if csv_path.name.endswith(".bak") or csv_path.name.endswith("_with_transition.csv"):
            continue
        matched.append(csv_path)

csv_files = sorted(set(matched))
print(f"Found {len(csv_files)} summary files under:\n  {ROOT_DIR}")
for pattern in SUMMARY_PATTERNS:
    print(f"  {pattern}: {sum(1 for p in csv_files if p.match(pattern))}")

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No lifetime or C-rate summary files matched under "
        f"{ROOT_DIR}. Run A01 or A02 first, then rerun this notebook."
    )

# Required columns use the saved-output schema: x100 = x_n,100,
# y100 = x_p,100, si_scale_a = s_V, and si_shift_b = U_off.
PARAM_COLS_FOR_TRANSITION = [
    "Cn_Si", "Cn_Gr", "x100", "Cp", "y100", "si_scale_a", "si_shift_b"
]


def safe_tag_part(value) -> str:
    """Return a filesystem-safe tag component for plot names."""
    text = str(value)
    for old, new in (("/", "by"), (" ", ""), (".", "p"), (":", "-")):
        text = text.replace(old, new)
    return "".join(ch for ch in text if ch.isalnum() or ch in {"_", "-"})


def transition_plot_tag(row, row_index: int) -> str:
    """Build a globally unique, readable tag for lifetime and C-rate rows.

    C-rate summaries can reuse segment_id across different input files, so the
    row index is always included to prevent later rows from overwriting earlier
    transition-SoC plots.
    """
    parts = [f"row_{row_index:03d}"]
    if "rpt_seq" in row.index and pd.notna(row["rpt_seq"]):
        parts.append(f"rptseq_{int(row['rpt_seq']):03d}")
    if "segment_id" in row.index and pd.notna(row["segment_id"]):
        parts.append(f"segment_{int(row['segment_id']):03d}")
    if "crate_label" in row.index and pd.notna(row["crate_label"]):
        parts.append("crate_" + safe_tag_part(row["crate_label"]))
    if "file" in row.index and pd.notna(row["file"]):
        parts.append(Path(str(row["file"])).stem[:40])
    elif "rpt_key" in row.index and pd.notna(row["rpt_key"]):
        parts.append("rptkey_" + safe_tag_part(row["rpt_key"]))
    return "_".join(safe_tag_part(part) for part in parts if part)


success_files = []
failed_files = []

for i, csv_path in enumerate(csv_files, 1):
    print(f"\n[{i}/{len(csv_files)}] Processing: {csv_path}")

    try:
        df = pd.read_csv(csv_path)
        missing_param_cols = [c for c in PARAM_COLS_FOR_TRANSITION if c not in df.columns]
        if missing_param_cols:
            raise KeyError(f"Missing required eSOH columns: {missing_param_cols}")

        out_df = append_transition_soc_to_summary_df(
            df,
            Qdata_expand=Qdata_expand,
            ocp_3=ocp_3,
            smooth_then_grad=smooth_then_grad,
            build_effective_si_ocp=build_effective_si_ocp,
            Gr_OCP_smooth=Gr_OCP_smooth,
            SI_RECONSTRUCTION_MODE=SI_RECONSTRUCTION_MODE,
            win=WIN,
            poly=POLY,
            threshold=THRESHOLD,
            persistence_window=PERSISTENCE_WINDOW,
            tol=TOL,
        )

        n_ok = out_df["transition_soc"].notna().sum() if "transition_soc" in out_df.columns else 0
        print(f"  Rows: {len(out_df)}, transition_soc computed for {n_ok} rows")

        if SAVE_TRANSITION_PLOTS:
            plot_dir = csv_path.parent / "transition_soc_plots"
            plot_dir.mkdir(parents=True, exist_ok=True)
            # Remove stale plots for this summary file so reruns produce one
            # image per row without retaining older naming conventions.
            for old_plot in plot_dir.glob(f"{csv_path.stem}_*_transition_soc.png"):
                old_plot.unlink()
            n_plot_ok = 0
            n_plot_fail = 0

            for ridx, row in out_df.iterrows():
                try:
                    row_for_transition = row[PARAM_COLS_FOR_TRANSITION].copy()
                    for col in PARAM_COLS_FOR_TRANSITION:
                        row_for_transition[col] = pd.to_numeric(row_for_transition[col], errors="coerce")

                    result_dict = compute_si_current_share_and_transition_7p(
                        row_for_transition,
                        Qdata_expand=Qdata_expand,
                        ocp_3=ocp_3,
                        smooth_then_grad=smooth_then_grad,
                        build_effective_si_ocp=build_effective_si_ocp,
                        Gr_OCP_smooth=Gr_OCP_smooth,
                        SI_RECONSTRUCTION_MODE=SI_RECONSTRUCTION_MODE,
                        win=WIN,
                        poly=POLY,
                        threshold=THRESHOLD,
                        persistence_window=PERSISTENCE_WINDOW,
                        tol=TOL,
                    )

                    tag = transition_plot_tag(row, ridx)
                    fig_path = plot_dir / f"{csv_path.stem}_{tag}_transition_soc.png"
                    title = f"{csv_path.stem} | {tag}"
                    if "Ah_throughput" in row.index and pd.notna(row["Ah_throughput"]):
                        title += f" | AhTh={row['Ah_throughput']:.1f}"
                    if "crate_label" in row.index and pd.notna(row["crate_label"]):
                        title += f" | {row['crate_label']}"

                    save_transition_soc_plot(
                        result_dict,
                        save_path=fig_path,
                        title=title,
                        threshold=THRESHOLD,
                    )
                    n_plot_ok += 1

                except Exception as e_plot:
                    n_plot_fail += 1
                    print(f"    Plot failed for row {ridx}: {e_plot}")

            print(f"  transition plots saved: {n_plot_ok}, failed: {n_plot_fail}")

        if not DRY_RUN:
            if OVERWRITE_CSV:
                if SAVE_BACKUP:
                    backup_path = csv_path.with_suffix(csv_path.suffix + ".bak")
                    if not backup_path.exists():
                        csv_path.replace(backup_path)
                    out_df.to_csv(csv_path, index=False)
                else:
                    out_df.to_csv(csv_path, index=False)
            else:
                new_path = csv_path.with_name(csv_path.stem + "_with_transition.csv")
                out_df.to_csv(new_path, index=False)
                print(f"  Saved to: {new_path}")

        success_files.append(csv_path)

    except Exception as e:
        print(f"  FAILED: {e}")
        traceback.print_exc(limit=1)
        failed_files.append((csv_path, repr(e)))

print("\n" + "=" * 60)
print(f"Done. Success: {len(success_files)} | Failed: {len(failed_files)}")

if failed_files:
    print("\nFailed files:")
    for path, err in failed_files:
        print(f"  - {path}\n    {err}")
